# makemore, part 1: the bigram model, from scratch

This is the *Practice* step of `unit_02_makemore_bigram.md`. Do the Cold Attempt there first.

Work top to bottom. Each milestone is one cell of stubs followed by a grader cell.
The grader stops at your first failure so there is always exactly one thing in front of you.

**Rules of engagement**
- Don't open the lecture. Don't open the makemore repo or the lecture notebook.
- Stuck on an *idea* for 20 min → ask the coaching chat for a hint.
- Stuck on *PyTorch syntax* → ask immediately, zero learning value in that.
- **Before you run a grader cell, say out loud what you expect to happen.**

The boilerplate (reading the file, the vocabulary, plotting, tensor plumbing, the sampling loop)
is written for you. Every function with `raise NotImplementedError` is the idea.

In [ ]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from test_makemore_bigram import grade

## Boilerplate — the data and the vocabulary

Nothing to write here. Run it, look at the shapes, move on.

The vocabulary is the 26 letters plus one boundary token `'.'` at index 0, which marks
both "the word starts here" and "the word ends here".

In [ ]:
words = open('../data/names.txt').read().splitlines()
print(len(words), words[:5])

chars = sorted(set(''.join(words)))
stoi = {s: i + 1 for i, s in enumerate(chars)}
stoi['.'] = 0
itos = {i: s for s, i in stoi.items()}
V = len(stoi)  # 27
print(V, itos)

## Milestone 1 — the count table

`N[i, j]` = how many times character `j` followed character `i` across every name,
counting the boundary on both ends of each word.

In [ ]:
def bigram_counts(words, stoi):
    """Count every (previous, next) character pair in the dataset.

    words: list of str, e.g. ['emma', 'olivia', ...]
    stoi:  dict char -> int, with '.' -> 0 as the boundary token
    Returns N, an integer tensor of shape (V, V) where N[stoi[a], stoi[b]] is the
    number of times b came right after a. Each word contributes len(word) + 1 pairs.
    """
    raise NotImplementedError

In [ ]:
def plot_counts(N, itos):
    """Plumbing: the 27x27 grid from the lecture."""
    plt.figure(figsize=(16, 16))
    plt.imshow(N, cmap='Blues')
    for i in range(N.shape[0]):
        for j in range(N.shape[1]):
            plt.text(j, i, itos[i] + itos[j], ha='center', va='bottom', color='gray', fontsize=8)
            plt.text(j, i, int(N[i, j]), ha='center', va='top', color='gray', fontsize=8)
    plt.axis('off')

# N = bigram_counts(words, stoi)
# plot_counts(N, itos)

In [ ]:
grade(bigram_counts, upto=1)

## Milestone 2 — rows become probabilities, then sample

Every row of `N` is "given this previous character, what came next". Turn each row into
a probability distribution. The sampling loop is given: it is `torch.multinomial` plumbing.
The grader seeds the generator and compares your samples to a reference, so the *numbers*
in `P` have to be right, not just roughly right.

In [ ]:
def counts_to_probs(N):
    """N: (V, V) integer counts.
    Returns P, a float tensor of shape (V, V) where every ROW sums to 1:
    P[i, j] = probability that j follows i.
    """
    raise NotImplementedError


def sample_names(P, itos, g, n=10):
    """Plumbing: draw n names from a (V, V) table of next-character probabilities.

    Start at the boundary token, keep drawing the next character from the row of the
    current one, stop when the boundary comes back out.
    """
    out = []
    for _ in range(n):
        ix = 0
        s = ''
        while True:
            ix = torch.multinomial(P[ix], 1, replacement=True, generator=g).item()
            if ix == 0:
                break
            s += itos[ix]
        out.append(s)
    return out

# g = torch.Generator().manual_seed(2147483647)
# P = counts_to_probs(bigram_counts(words, stoi))
# sample_names(P, itos, g)

In [ ]:
grade(bigram_counts, counts_to_probs, upto=2)

## Milestone 3 — the number to minimize

This is the unit's core question. A model hands you a probability distribution over the next
character for every example. Reduce "how well did it do on the whole dataset" to one number
that is *small when the model is good*.

Contract: `probs` has one row per example, `ys` is the character that actually came next.
For the counting model, the row for example `i` is `P[xs[i]]`, i.e. `probs = P[xs]`.
The `(xs, ys)` tensor construction is given below.

In [ ]:
def build_dataset(words, stoi):
    """Plumbing: every bigram in the dataset as two aligned int tensors.
    xs[k] is the previous character's index, ys[k] the next one's. Shape (num_pairs,)."""
    xs, ys = [], []
    for w in words:
        chs = ['.'] + list(w) + ['.']
        for a, b in zip(chs, chs[1:]):
            xs.append(stoi[a])
            ys.append(stoi[b])
    return torch.tensor(xs), torch.tensor(ys)

xs, ys = build_dataset(words, stoi)
print(xs.shape, ys.shape, xs[:8], ys[:8])


def nll(probs, ys):
    """Average negative log likelihood.

    probs: (n, V) float, row i is a model's distribution over the next character
           for example i (each row sums to 1)
    ys:    (n,) int, the character that actually came next for example i
    Returns a scalar tensor. Lower is better; a model that gives every actual next
    character probability 1 scores 0.
    """
    raise NotImplementedError

In [ ]:
grade(bigram_counts, counts_to_probs, nll, upto=3)

## Milestone 4 — the same table, but as a neural net

One layer of 27 neurons, no bias, no nonlinearity. `F.softmax` and `F.cross_entropy` are
off limits in this notebook: the point is to see what softmax is made of.

**forward(W, xs)**
- Inputs: `W` is `(V, V)` float with `requires_grad=True`; row `i` is the vector of logits
  for previous character `i`. `xs` is `(n,)` int, previous-character indices.
- Computes `probs = exp(logits) / (row sums of exp(logits))` with
  `logits = one_hot(xs, V) @ W`, i.e. torch's `softmax(one_hot(xs).float() @ W, dim=1)`.
- Returns `probs` of shape `(n, V)`: every entry in `[0, 1]`, every row sums to 1, and
  `requires_grad` is `True` (still attached to `W`). For `n == 1` the shape is `(1, V)`,
  not `(V,)`. Row `k` depends only on `xs[k]`.

The grader checks, in order: no dtype error in the matmul → shape `(64, V)` → entries in
`[0, 1]` → rows sum to 1 → values match torch's softmax → `requires_grad` → gradient
w.r.t. `W` matches torch's → batch rows equal one-character-at-a-time rows.

In [ ]:
def forward(W, xs):
    """The one-layer 'net'.

    W:  (V, V) float, requires_grad. Row i is the vector of logits for previous char i.
    xs: (n,) int, previous-character indices
    Returns probs of shape (n, V), each row a distribution over the next character,
    still attached to W in the autograd graph.
    """
    raise NotImplementedError

In [ ]:
grade(bigram_counts, counts_to_probs, nll, forward, upto=4)

## Milestone 5 — gradient descent, and the punchline

`W` is created for you below. `lr=50` is not a typo for this problem. The grader trains
for 100 steps and then compares your loss to the counting model's: the same answer,
reached two ways.

**train(W, xs, ys, steps, lr)**
- Inputs: `W` is a `(V, V)` float leaf with `requires_grad=True`, modified **in place**
  (the grader keeps a reference to the very same tensor). `xs`, `ys` are `(n,)` int, the
  whole dataset.
- Each step computes `loss = nll(forward(W, xs), ys)` over the full dataset and moves
  `W` by `-lr * dloss/dW`.
- Returns a Python list of exactly `steps` floats. `losses[k]` is the loss of `W` as it
  was at the start of step `k`, before that step's update. So `losses[0]` is the loss of
  the untouched `W`, and the loss of the returned `W` is at most `losses[-1]`.

The grader checks, in order: the update raises no autograd error → a list of 100 floats
→ `W` actually changed → no nan/inf → `losses[0]` equals the loss of the initial `W` →
the loss did not go up → final loss below 2.6 → the final `W`'s loss is ≤ `losses[-1]` →
final loss within 0.05 of the counting model's (≈2.454).

In [ ]:
g = torch.Generator().manual_seed(2147483647)
W = torch.randn((V, V), generator=g, requires_grad=True)


def train(W, xs, ys, steps, lr):
    """Plain gradient descent on W, in place.

    W:      (V, V) float, requires_grad=True. Modified in place.
    xs, ys: (n,) int, the whole dataset
    Returns the list of per-step losses (as Python floats), one per step, each recorded
    BEFORE that step's update.
    """
    raise NotImplementedError

# losses = train(W, xs, ys, steps=100, lr=50.0)
# plt.plot(losses)

In [ ]:
grade(bigram_counts, counts_to_probs, nll, forward, train, upto=5)

## Milestone 6 — the learned count table

If you feed the net every character in order, the 27 rows you get back are a `(V, V)`
table exactly like `P` from milestone 2, and `sample_names` works on it unchanged.

**neural_probs(W)**
- Input: `W` is `(V, V)` float; it may or may not require grad.
- Computes `forward(W, xs)` for `xs = [0, 1, ..., V-1]`, in that order.
- Returns a `(V, V)` tensor where row `i` equals `forward(W, tensor([i]))[0]`. Rows sum
  to 1. Do not detach it; callers that need a detached table call `.detach()` themselves.

The grader checks, in order: shape `(V, V)` → equals torch's `softmax(W, dim=1)` → with
`W = log(P_counts)` the seeded samples equal the counting model's names, bit for bit →
samples from milestone 5's trained `W` contain only a–z → their average length is
between 3 and 12.

In [ ]:
def neural_probs(W):
    """W: (V, V). Returns the (V, V) table where row i is the net's distribution over
    the next character given previous character i."""
    raise NotImplementedError

# g = torch.Generator().manual_seed(2147483647)
# sample_names(neural_probs(W).detach(), itos, g)

In [ ]:
grade(bigram_counts, counts_to_probs, nll, forward, train, neural_probs, upto=6)

## Milestone 7 (stretch) — smoothing and regularization are the same idea

Two ways to stop the model from being *certain*: add fake counts to the table, or penalize
big weights in the net. Write both. The grader checks that both, pushed to the extreme,
flatten the table to uniform.

No later milestone needs anything from this one, so the final grade cell below passes
`skip=(7,)`. Remove the 7 once you write these two functions.

**smooth_counts(N, k)**
- Inputs: `N` is `(V, V)` integer counts; `k` is a non-negative number, the fake count
  added to every cell.
- Computes the milestone-2 probability table of `N + k`.
- Returns a `(V, V)` float tensor whose rows sum to 1. `k == 0` is exactly the counting
  model; `k > 0` leaves no zero anywhere; `k → ∞` is uniform, `1/V` everywhere.

**nll_reg(W, xs, ys, alpha)**
- Inputs: `W` is `(V, V)` float with `requires_grad=True`; `xs`, `ys` are `(n,)` int;
  `alpha` is a non-negative float.
- Computes `nll(forward(W, xs), ys) + alpha * mean(W ** 2)`. Mean, not sum.
- Returns a scalar tensor attached to `W`. `alpha == 0` is exactly the milestone-3 loss.

The grader checks, in order: `k=0` matches the counting model → rows sum to 1 at `k=1` →
no zeros at `k=1` → `k=1` costs some likelihood on the training data → `k=1e7` is
uniform → `alpha=0` equals plain nll → `alpha=0.1` value (mean, not sum) →
`requires_grad` → 60 steps at `alpha=50` drive `neural_probs` to uniform.

In [ ]:
def smooth_counts(N, k):
    """N: (V, V) integer counts, k: a non-negative fake count added to every cell.
    Returns the (V, V) probability table of the smoothed counts (rows sum to 1)."""
    raise NotImplementedError


def nll_reg(W, xs, ys, alpha):
    """The milestone-3 loss of forward(W, xs) against ys, plus alpha times the
    mean of W squared. Returns a scalar tensor attached to W."""
    raise NotImplementedError

In [ ]:
grade(bigram_counts, counts_to_probs, nll, forward, train, neural_probs, smooth_counts, nll_reg, skip=(7,))

## Your own sampling-and-training loop

From memory, no scrolling up: start from a fresh random `W`, train it, and print ten names
from it next to ten names from the counting model using the same seed. Then say, in a
comment, why they do or don't match.

In [ ]:
g = torch.Generator().manual_seed(2147483647)
W2 = torch.randn((V, V), generator=g, requires_grad=True)

for step in range(200):
    ...


## Scratch

Space to poke at things. `plot_counts(N, itos)` draws the grid; `sample_names(P, itos, g)` draws names.